# Comprehensive Model Evaluation and Performance Analysis

This notebook provides detailed evaluation of the CNN product classification model with:
- Performance metrics and statistical analysis
- Confusion matrices and classification reports
- Error analysis and model interpretability
- Inference demonstrations with real examples
- Benchmarking and comparison results

## Table of Contents
1. [Setup and Configuration](#setup)
2. [Model Loading and Validation](#loading)
3. [Test Data Preparation](#data-prep)
4. [Performance Metrics](#metrics)
5. [Detailed Analysis](#analysis)
6. [Inference Demonstrations](#inference)
7. [Model Interpretability](#interpretability)
8. [Benchmarking Results](#benchmarking)
9. [Conclusions and Recommendations](#conclusions)

## 1. Setup and Configuration {#setup}

In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path
sys.path.append('..')

# Import our modular components
from config import get_config
from services.ml_models.cnn_classifier import CNNClassifier
from services.ml_models.evaluation_framework import ModelEvaluator
from services.data_processing.preprocessing_pipeline import ImagePreprocessor
from utils.logging_config import get_logger

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Initialize configuration and logging
config = get_config()
logger = get_logger('evaluation_notebook')

print("✅ Setup completed successfully!")
print(f"📊 Evaluation started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Model Loading and Validation {#loading}

In [ ]:
# Initialize and load the CNN model
print("🔄 Loading CNN model...")

cnn_model = CNNClassifier()
model_path = config.model.cnn_model_path

# Try to load existing model, create demo model if not available
if os.path.exists(model_path):
    success = cnn_model.load_model(model_path)
    print(f"✅ Model loaded from {model_path}")
else:
    print(f"⚠️  Model file not found at {model_path}")
    print("🔄 Creating demo model for evaluation...")
    success = cnn_model._create_demo_model()
    if success:
        cnn_model._is_loaded = True
        print("✅ Demo model created successfully")

# Display model information
if cnn_model.is_loaded:
    model_info = cnn_model.get_model_info()
    print("\n📋 Model Information:")
    print(f"  Name: {model_info.name}")
    print(f"  Version: {model_info.version}")
    print(f"  Type: {model_info.model_type}")
    print(f"  Input Shape: {model_info.input_shape}")
    print(f"  Output Shape: {model_info.output_shape}")
    print(f"  Classes: {len(cnn_model.class_names)}")
    print(f"  Class Names: {cnn_model.class_names}")
else:
    print("❌ Failed to load or create model")
    raise RuntimeError("Model initialization failed")

## 3. Test Data Preparation {#data-prep}

In [ ]:
# Create synthetic test data for evaluation
print("🔄 Preparing test data...")

# Generate synthetic test samples for each class
np.random.seed(42)  # For reproducibility

def create_synthetic_image(class_name: str, image_size: tuple = (224, 224, 3)) -> np.ndarray:
    """Create a synthetic image with class-specific characteristics."""
    # Create base random image
    image = np.random.randint(0, 256, image_size, dtype=np.uint8)
    
    # Add class-specific patterns
    if 'computer' in class_name or 'electronics' in class_name:
        # Add rectangular patterns for tech items
        image[50:150, 50:150] = [100, 100, 150]  # Blue-ish rectangle
    elif 'clothing' in class_name or 't-shirt' in class_name:
        # Add fabric-like texture
        image[::2, ::2] = [200, 150, 100]  # Fabric pattern
    elif 'kitchen' in class_name or 'teapot' in class_name:
        # Add circular patterns for kitchen items
        center = (image_size[0]//2, image_size[1]//2)
        y, x = np.ogrid[:image_size[0], :image_size[1]]
        mask = (x - center[0])**2 + (y - center[1])**2 <= 50**2
        image[mask] = [150, 100, 50]  # Brown circle
    elif 'antique' in class_name:
        # Add vintage-like coloring
        image = (image * 0.8 + 50).astype(np.uint8)  # Darker, vintage look
    
    return image

# Create test dataset
test_images = []
test_labels = []
samples_per_class = 20

for class_idx, class_name in enumerate(cnn_model.class_names):
    for i in range(samples_per_class):
        # Create synthetic image
        image = create_synthetic_image(class_name)
        test_images.append(image)
        test_labels.append(class_name)

print(f"✅ Created {len(test_images)} test samples")
print(f"📊 Classes: {len(set(test_labels))}")
print(f"📊 Samples per class: {samples_per_class}")

# Display class distribution
class_counts = pd.Series(test_labels).value_counts()
print("\n📈 Test Data Distribution:")
for class_name, count in class_counts.items():
    print(f"  {class_name}: {count} samples")

## 4. Performance Metrics {#metrics}

In [ ]:
# Initialize model evaluator
print("🔄 Running comprehensive model evaluation...")

evaluator = ModelEvaluator(cnn_model, task_type='classification')

# Run evaluation
evaluation_results = evaluator.evaluate(
    test_data=test_images,
    test_labels=test_labels,
    batch_size=16
)

print("✅ Evaluation completed!")

# Display key metrics
metrics = evaluation_results['metrics']
print("\n📊 Performance Metrics:")
print("=" * 40)
for metric_name, score in metrics.items():
    print(f"{metric_name:20s}: {score:.4f}")

# Create metrics visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Model Performance Metrics', fontsize=16, fontweight='bold')

# Accuracy comparison
accuracy_metrics = {k: v for k, v in metrics.items() if 'accuracy' in k}
if accuracy_metrics:
    axes[0, 0].bar(accuracy_metrics.keys(), accuracy_metrics.values(), color='skyblue')
    axes[0, 0].set_title('Accuracy Metrics')
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].tick_params(axis='x', rotation=45)

# Precision metrics
precision_metrics = {k: v for k, v in metrics.items() if 'precision' in k}
if precision_metrics:
    axes[0, 1].bar(precision_metrics.keys(), precision_metrics.values(), color='lightgreen')
    axes[0, 1].set_title('Precision Metrics')
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].tick_params(axis='x', rotation=45)

# Recall metrics
recall_metrics = {k: v for k, v in metrics.items() if 'recall' in k}
if recall_metrics:
    axes[1, 0].bar(recall_metrics.keys(), recall_metrics.values(), color='lightcoral')
    axes[1, 0].set_title('Recall Metrics')
    axes[1, 0].set_ylim(0, 1)
    axes[1, 0].tick_params(axis='x', rotation=45)

# F1 metrics
f1_metrics = {k: v for k, v in metrics.items() if 'f1' in k}
if f1_metrics:
    axes[1, 1].bar(f1_metrics.keys(), f1_metrics.values(), color='gold')
    axes[1, 1].set_title('F1 Score Metrics')
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Save metrics plot
os.makedirs('../static/images/evaluation', exist_ok=True)
plt.savefig('../static/images/evaluation/performance_metrics.png', dpi=300, bbox_inches='tight')
print("📊 Metrics visualization saved to ../static/images/evaluation/performance_metrics.png")

## 5. Detailed Analysis {#analysis}

In [ ]:
# Analyze evaluation results in detail
analysis = evaluation_results['analysis']

print("🔍 Detailed Analysis Results:")
print("=" * 50)

# Confidence statistics
if 'confidence_stats' in analysis:
    conf_stats = analysis['confidence_stats']
    print("\n📈 Confidence Statistics:")
    print(f"  Mean Confidence: {conf_stats['mean']:.4f}")
    print(f"  Std Deviation: {conf_stats['std']:.4f}")
    print(f"  Min Confidence: {conf_stats['min']:.4f}")
    print(f"  Max Confidence: {conf_stats['max']:.4f}")
    print(f"  Median Confidence: {conf_stats['median']:.4f}")

# Error analysis
if 'error_analysis' in analysis:
    error_analysis = analysis['error_analysis']
    print(f"\n❌ Error Analysis:")
    print(f"  Total Errors: {error_analysis['num_errors']}")
    print(f"  Error Rate: {error_analysis['error_rate']:.4f} ({error_analysis['error_rate']*100:.2f}%)")
    print(f"  Accuracy: {1-error_analysis['error_rate']:.4f} ({(1-error_analysis['error_rate'])*100:.2f}%)")

# Class distribution analysis
if 'class_distribution' in analysis:
    class_dist = analysis['class_distribution']
    print("\n📊 Class Distribution Analysis:")
    
    # Create comparison plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # True labels distribution
    true_classes = list(class_dist['true'].keys())
    true_counts = list(class_dist['true'].values())
    ax1.bar(true_classes, true_counts, color='skyblue', alpha=0.7)
    ax1.set_title('True Label Distribution')
    ax1.set_xlabel('Classes')
    ax1.set_ylabel('Count')
    ax1.tick_params(axis='x', rotation=45)
    
    # Predicted labels distribution
    pred_classes = list(class_dist['predicted'].keys())
    pred_counts = list(class_dist['predicted'].values())
    ax2.bar(pred_classes, pred_counts, color='lightcoral', alpha=0.7)
    ax2.set_title('Predicted Label Distribution')
    ax2.set_xlabel('Classes')
    ax2.set_ylabel('Count')
    ax2.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Save distribution plot
    plt.savefig('../static/images/evaluation/class_distribution.png', dpi=300, bbox_inches='tight')
    print("📊 Class distribution plot saved")

## 6. Inference Demonstrations {#inference}

In [ ]:
# Demonstrate model inference with step-by-step process
print("🔄 Running Inference Demonstrations...")
print("=" * 50)

# Select sample images for demonstration
demo_indices = [0, 20, 40, 60, 80]  # One from each class type
demo_results = []

for i, idx in enumerate(demo_indices):
    print(f"\n🔍 Demo {i+1}: Processing sample {idx}")
    
    # Get sample
    sample_image = test_images[idx]
    true_label = test_labels[idx]
    
    print(f"  True Label: {true_label}")
    print(f"  Image Shape: {sample_image.shape}")
    
    # Step 1: Preprocessing
    print("  Step 1: Preprocessing image...")
    preprocessed = cnn_model.preprocess_input(sample_image)
    if preprocessed is not None:
        print(f"    ✅ Preprocessed shape: {preprocessed.shape}")
    else:
        print("    ❌ Preprocessing failed")
        continue
    
    # Step 2: Model prediction
    print("  Step 2: Running model inference...")
    try:
        prediction_result = cnn_model.predict(sample_image)
        print(f"    ✅ Prediction: {prediction_result.prediction}")
        print(f"    ✅ Confidence: {prediction_result.confidence:.4f}")
        print(f"    ✅ Processing Time: {prediction_result.processing_time:.4f}s")
        
        # Step 3: Result analysis
        print("  Step 3: Analyzing results...")
        is_correct = prediction_result.prediction == true_label
        print(f"    Correct Prediction: {'✅ Yes' if is_correct else '❌ No'}")
        
        if prediction_result.metadata and 'top_predictions' in prediction_result.metadata:
            top_preds = prediction_result.metadata['top_predictions'][:3]
            print("    Top 3 Predictions:")
            for j, pred in enumerate(top_preds, 1):
                print(f"      {j}. {pred['class']}: {pred['confidence']:.4f}")
        
        demo_results.append({
            'sample_idx': idx,
            'true_label': true_label,
            'predicted_label': prediction_result.prediction,
            'confidence': prediction_result.confidence,
            'correct': is_correct,
            'processing_time': prediction_result.processing_time
        })
        
    except Exception as e:
        print(f"    ❌ Prediction failed: {e}")

# Summary of demonstrations
print("\n📋 Inference Demonstration Summary:")
print("=" * 40)
if demo_results:
    correct_demos = sum(1 for r in demo_results if r['correct'])
    avg_confidence = np.mean([r['confidence'] for r in demo_results])
    avg_time = np.mean([r['processing_time'] for r in demo_results if r['processing_time']])
    
    print(f"Total Demonstrations: {len(demo_results)}")
    print(f"Correct Predictions: {correct_demos}/{len(demo_results)} ({correct_demos/len(demo_results)*100:.1f}%)")
    print(f"Average Confidence: {avg_confidence:.4f}")
    print(f"Average Processing Time: {avg_time:.4f}s")
else:
    print("No successful demonstrations")

## 7. Model Interpretability {#interpretability}

In [ ]:
# Model interpretability and explanation analysis
print("🔍 Model Interpretability Analysis")
print("=" * 40)

# Analyze prediction patterns
if demo_results:
    print("\n📊 Prediction Pattern Analysis:")
    
    # Confidence vs Accuracy correlation
    confidences = [r['confidence'] for r in demo_results]
    accuracies = [1 if r['correct'] else 0 for r in demo_results]
    
    if len(confidences) > 1:
        correlation = np.corrcoef(confidences, accuracies)[0, 1]
        print(f"  Confidence-Accuracy Correlation: {correlation:.4f}")
        
        if correlation > 0.5:
            print("    ✅ Strong positive correlation - model confidence is reliable")
        elif correlation > 0.2:
            print("    ⚠️  Moderate correlation - confidence somewhat reliable")
        else:
            print("    ❌ Weak correlation - confidence may not be reliable")

# Feature importance analysis (simplified)
print("\n🎯 Model Decision Analysis:")
print("  The CNN model makes decisions based on:")
print("  1. Low-level features: edges, textures, colors")
print("  2. Mid-level features: shapes, patterns, object parts")
print("  3. High-level features: complete objects, spatial relationships")

# Class-specific insights
print("\n🏷️  Class-Specific Model Insights:")
class_insights = {
    'computer': 'Likely focuses on rectangular shapes, screen-like patterns, and tech-related textures',
    'electronics': 'Detects electronic components, circuits, and modern device characteristics',
    't-shirt': 'Recognizes fabric textures, clothing shapes, and apparel-specific features',
    'clothing': 'Identifies various clothing items through fabric patterns and garment shapes',
    'kitchen': 'Focuses on kitchen-specific shapes, utensil patterns, and cooking-related objects',
    'teapot': 'Detects rounded shapes, spout-like features, and ceramic/metal textures',
    'antique_car': 'Recognizes vintage automotive features, classic car shapes, and aged textures',
    'automotive': 'Identifies vehicle-related components, mechanical parts, and automotive textures',
    'home_garden': 'Detects home/garden items through decorative patterns and outdoor textures',
    'office': 'Recognizes office supplies, business items, and workplace-related objects'
}

for class_name in cnn_model.class_names:
    if class_name in class_insights:
        print(f"  {class_name}: {class_insights[class_name]}")

# Model limitations and considerations
print("\n⚠️  Model Limitations and Considerations:")
print("  1. Synthetic data training may not generalize to real-world images")
print("  2. Limited training data may cause overfitting to specific patterns")
print("  3. Class imbalance could affect prediction accuracy for some categories")
print("  4. Model may be sensitive to image quality, lighting, and orientation")
print("  5. Cross-domain generalization (e.g., different image styles) may be limited")

## 8. Benchmarking Results {#benchmarking}

In [ ]:
# Performance benchmarking and comparison
print("⚡ Performance Benchmarking")
print("=" * 30)

# Benchmark inference speed
print("\n🚀 Inference Speed Benchmark:")
import time

# Test different batch sizes
batch_sizes = [1, 4, 8, 16]
benchmark_results = {}

for batch_size in batch_sizes:
    print(f"\n  Testing batch size: {batch_size}")
    
    # Prepare batch
    batch_images = test_images[:batch_size]
    
    # Warm-up run
    for img in batch_images[:min(2, len(batch_images))]:
        try:
            cnn_model.predict(img)
        except:
            pass
    
    # Benchmark runs
    times = []
    for run in range(5):  # 5 runs for averaging
        start_time = time.time()
        
        for img in batch_images:
            try:
                cnn_model.predict(img)
            except:
                pass
        
        end_time = time.time()
        times.append(end_time - start_time)
    
    avg_time = np.mean(times)
    std_time = np.std(times)
    throughput = batch_size / avg_time
    
    benchmark_results[batch_size] = {
        'avg_time': avg_time,
        'std_time': std_time,
        'throughput': throughput
    }
    
    print(f"    Average time: {avg_time:.4f}s (±{std_time:.4f}s)")
    print(f"    Throughput: {throughput:.2f} images/second")

# Visualize benchmark results
if benchmark_results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    batch_sizes_list = list(benchmark_results.keys())
    avg_times = [benchmark_results[bs]['avg_time'] for bs in batch_sizes_list]
    throughputs = [benchmark_results[bs]['throughput'] for bs in batch_sizes_list]
    
    # Processing time vs batch size
    ax1.plot(batch_sizes_list, avg_times, 'o-', linewidth=2, markersize=8)
    ax1.set_xlabel('Batch Size')
    ax1.set_ylabel('Average Processing Time (s)')
    ax1.set_title('Processing Time vs Batch Size')
    ax1.grid(True, alpha=0.3)
    
    # Throughput vs batch size
    ax2.plot(batch_sizes_list, throughputs, 'o-', color='orange', linewidth=2, markersize=8)
    ax2.set_xlabel('Batch Size')
    ax2.set_ylabel('Throughput (images/second)')
    ax2.set_title('Throughput vs Batch Size')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Save benchmark plot
    plt.savefig('../static/images/evaluation/benchmark_results.png', dpi=300, bbox_inches='tight')
    print("\n📊 Benchmark results saved")

# Memory usage estimation
print("\n💾 Memory Usage Analysis:")
model_info = cnn_model.get_model_info()
if model_info.parameters:
    print(f"  Model parameters: {model_info.parameters.get('num_classes', 'N/A')} classes")
print(f"  Input shape: {model_info.input_shape}")
print(f"  Estimated memory per image: ~{np.prod(model_info.input_shape) * 4 / 1024:.2f} KB")
print(f"  Batch processing recommended for: >10 images")

## 9. Conclusions and Recommendations {#conclusions}

In [ ]:
# Generate comprehensive evaluation report
print("📋 Generating Comprehensive Evaluation Report")
print("=" * 50)

# Generate and display the report
report = evaluator.generate_report()
print(report)

# Save detailed results
results_dir = Path('../data/evaluation_results')
results_dir.mkdir(exist_ok=True)

# Save evaluation results as JSON
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_file = results_dir / f'evaluation_results_{timestamp}.json'
evaluator.save_results(str(results_file))

# Save report as text
report_file = results_dir / f'evaluation_report_{timestamp}.txt'
with open(report_file, 'w') as f:
    f.write(report)

print(f"\n💾 Results saved to:")
print(f"  📊 JSON: {results_file}")
print(f"  📄 Report: {report_file}")

# Final recommendations
print("\n🎯 Key Recommendations:")
print("=" * 30)

# Performance-based recommendations
if 'accuracy' in metrics:
    accuracy = metrics['accuracy']
    if accuracy > 0.9:
        print("✅ Model Performance: EXCELLENT")
        print("   - Ready for production deployment")
        print("   - Consider A/B testing with real users")
    elif accuracy > 0.8:
        print("✅ Model Performance: GOOD")
        print("   - Suitable for production with monitoring")
        print("   - Consider additional training data")
    elif accuracy > 0.7:
        print("⚠️  Model Performance: FAIR")
        print("   - Needs improvement before production")
        print("   - Recommend data augmentation and hyperparameter tuning")
    else:
        print("❌ Model Performance: POOR")
        print("   - Requires significant improvements")
        print("   - Consider architecture changes and more training data")

print("\n🔧 Technical Recommendations:")
print("1. Data Quality:")
print("   - Collect more diverse, real-world training data")
print("   - Implement data validation and quality checks")
print("   - Balance class distributions")

print("\n2. Model Improvements:")
print("   - Experiment with transfer learning (ResNet, EfficientNet)")
print("   - Implement advanced data augmentation techniques")
print("   - Add regularization to prevent overfitting")

print("\n3. Production Readiness:")
print("   - Implement model versioning and rollback capabilities")
print("   - Set up continuous monitoring and alerting")
print("   - Create automated retraining pipelines")

print("\n4. Evaluation Process:")
print("   - Establish regular evaluation schedules")
print("   - Create test datasets from production data")
print("   - Implement A/B testing framework")

print("\n🎉 Evaluation Complete!")
print(f"📅 Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n📊 All evaluation artifacts have been saved for future reference.")